# Merge revision trên BIRD dev bằng vLLM (model merge của bài báo)

Notebook này chạy `cycloneboy/CscSQL-Merge-Qwen2.5-Coder-1.5B-Instruct` trên các cặp SQL mà pool 64 mẫu còn bất đồng: 8 lần sinh mỗi cặp, prompt theo CSC-SQL. Đánh giá chạy ở máy local bằng `evaluation.merge_study`.

**Chuẩn bị:**
1. Trên máy local: tạo `data/benchmarks/bird_dev_merge_pairs.jsonl` và đóng gói lại (lệnh ở README).
2. Kaggle: *New Version* của dataset, upload `dist/adaptive-slm-sql-kaggle.zip`.
3. Notebook này: *Add Input* dataset đó. *Settings*: **Accelerator = GPU T4 x2**, **Internet = On**.
4. Đặt `LIMIT = 20` chạy thử, rồi `LIMIT = None` và *Save Version → Save & Run All* (chạy nền, tối đa 12 giờ, tắt máy được).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cài vLLM vào venv Python 3.12 riêng (khoảng 3–5 phút). Image Kaggle hiện chạy Python 3.13, còn vLLM 0.8.5 chỉ có bản cho Python < 3.13. `uv` tự tải Python 3.12; `--exclude-newer` khóa mọi thư viện phụ (transformers, torch…) về thời điểm vLLM 0.8.5 ra mắt để không bị bản mới làm hỏng. Venv nằm ở `/tmp` nên không bị lưu vào *Output*.

T4 không có bfloat16 nên dùng float16; engine V0 chạy được trên T4.

In [ ]:
import glob, os, shutil, subprocess, sys
VENV = "/tmp/vllm-env"
PY = f"{VENV}/bin/python"


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Lỗi khi chạy: {cmd}")


if not os.path.exists(PY):
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv -q --python 3.12 {VENV}")
    sh(f'{sys.executable} -m uv pip install -q --python {PY} --exclude-newer 2025-05-15 '
       '"vllm==0.8.5.post1" "sqlglot==26.16.4" "PyYAML==6.0.2"')

In [ ]:
r = subprocess.run([PY, "-c", "import sys, vllm; print('python', sys.version.split()[0], '| vllm', vllm.__version__)"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0, "Không import được vllm trong venv"

Chép code và database sang `/tmp` (ghi được, không bị lưu vào *Output*). Database phải nằm ở ổ ghi được vì `card_games.sqlite` của BIRD ở chế độ WAL: SQLite cần tạo file `-shm` cạnh nó, nên mở trên `/kaggle/input` (chỉ đọc) sẽ báo `unable to open database file`.

In [ ]:
# Output của version trước (nếu được Add Input) cũng chứa một bản code; chỉ lấy bản có database thật.
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adaptive-slm-sql/config.yaml", recursive=True)
       if os.path.isdir(os.path.join(os.path.dirname(p), "data", "external"))
       and not os.path.islink(os.path.join(os.path.dirname(p), "data", "external"))]
assert src, "Không thấy adaptive-slm-sql/ trong /kaggle/input: kiểm tra Add Input"
src = src[0]
work = "/tmp/adaptive-slm-sql"  # ngoài /kaggle/working để Output chỉ có kết quả
if not os.path.exists(work):
    shutil.copytree(src, work, ignore=shutil.ignore_patterns("external"))
    shutil.copytree(os.path.join(src, "data", "external"), "/tmp/external")
    os.symlink("/tmp/external", os.path.join(work, "data", "external"))
os.chdir(work)
print(os.getcwd(), os.listdir("data/benchmarks"))

## Sinh merge

- Mỗi GPU xử lý một nửa số cặp, ghi vào `results/bird-dev-merge/shard0/`, `shard1/`.
- `LIMIT = 20` để thử nhanh trước, rồi đặt `None`.
- Tiến độ của từng GPU in mỗi phút; log đầy đủ ở `results/<RUN>/shard*.log`.
- Nếu có `merge.jsonl` từ version trước trong input thì nó được chép sang để chạy tiếp.

In [ ]:
import time

DATASET = "data/benchmarks/bird_dev.json"
PAIRS = "data/benchmarks/bird_dev_merge_pairs.jsonl"
RUN = "bird-dev-merge"
GPUS = 2
LIMIT = None  # 20 để thử nhanh
OUTPUT = f"/kaggle/working/results/{RUN}"
assert os.path.exists(PAIRS), "Thiếu file cặp SQL: đóng gói lại zip với --extra"

os.makedirs(OUTPUT, exist_ok=True)
procs = []
for shard in range(GPUS):
    out = f"{OUTPUT}/shard{shard}"
    previous = glob.glob(f"/kaggle/input/**/results/{RUN}/shard{shard}/merge.jsonl", recursive=True)
    if previous and not os.path.exists(f"{out}/merge.jsonl"):
        os.makedirs(out, exist_ok=True)
        shutil.copy(previous[0], f"{out}/merge.jsonl")
        print("Chạy tiếp từ", previous[0])
    cmd = [PY, "-u", "-m", "evaluation.vllm_merge", "run", "--dataset", DATASET, "--pairs", PAIRS,
           "--output", out, "--dtype", "half", "--shard", f"{shard}/{GPUS}"]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    env = dict(os.environ, VLLM_USE_V1="0", CUDA_VISIBLE_DEVICES=str(shard))
    log = f"{OUTPUT}/shard{shard}.log"
    with open(log, "w") as handle:
        procs.append((shard, log, subprocess.Popen(cmd, stdout=handle, stderr=subprocess.STDOUT, env=env)))

seen = [0] * GPUS
while True:
    running = any(proc.poll() is None for _, _, proc in procs)
    for shard, log, _ in procs:
        with open(log, encoding="utf-8", errors="replace") as handle:
            lines = handle.readlines()
        for line in lines[seen[shard]:]:
            if not line.startswith(("INFO", "WARNING", "Capturing", "Loading", "(")) or "Error" in line:
                print(f"[GPU {shard}]", line, end="")
        seen[shard] = len(lines)
    if not running:
        break
    time.sleep(60)
for shard, _, proc in procs:
    print(f"GPU {shard}: exit code", proc.returncode)

Kiểm tra nhanh: số cặp đã xong và tỷ lệ lần sinh chạy được, đúng.

In [ ]:
import json

for path in sorted(glob.glob(f"{OUTPUT}/**/merge.jsonl", recursive=True)):
    rows = [json.loads(line) for line in open(path, encoding="utf-8")]
    draws = [d for r in rows for d in r["draws"]]
    print(path, len(rows), "cặp,", len(draws), "lần sinh,",
          f"chạy được {sum(d['success'] for d in draws) / max(1, len(draws)):.1%},",
          f"đúng {sum(bool(d['correct_set']) for d in draws) / max(1, len(draws)):.1%}")

Nén kết quả để tải về (tab *Output* hoặc link bên dưới). Giải nén vào thư mục project local, giữ nguyên `results/<RUN>/` (không lồng `results/results/`).

In [ ]:
archive = shutil.make_archive(f"/kaggle/working/{RUN}", "zip", "/kaggle/working", f"results/{RUN}")
from IPython.display import FileLink
os.chdir("/kaggle/working")
FileLink(os.path.basename(archive))